# Calculate unit-wise difference between the Taunton-Worcester scenario and base

Aggregate differences by stroke unit.

# TO DO 

I don't think the results are right. Go back to unit-by-unit method?

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
dir_data = 'data'
dir_output = 'output'
# Scenario results:
file_results_comparison = 'results_full_comparison_taunworc_base.csv'
# Reference:
file_hospitals = 'stroke_hospitals_2025.csv'
# Output:

## Load data

### Scenario comparison results

In [3]:
p = os.path.join(dir_output, file_results_comparison)
df = pd.read_csv(p, index_col=0)

In [4]:
display(df.head(3))

,area_lower,Admissions_2025_Est,nearest_ivt_unit_base,transfer_unit_base,transfer_required_base,drip_ship_mt_travel_time_base,drip_ship_mrs_0-2_base,lvo_drip_ship_ivt_mt_mrs_0-2_base,drip_ship_mrs_0-2_transferloss0!6_base,lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base,...,drip_ship_mrs_0-2_taunworc,lvo_drip_ship_ivt_mt_mrs_0-2_taunworc,drip_ship_mrs_0-2_transferloss0!6_taunworc,lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_taunworc,change_in_drip_ship_mt_travel_time,change_in_drip_ship_mrs_0-2,change_in_lvo_drip_ship_ivt_mt_mrs_0-2,change_in_drip_ship_mrs_0-2_transferloss0!6,change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6,changed
LSOA,,,,,,,,,,,,,,,,,,,,,
ADUR001A,Adur 001A,1.49403,BN25BE,BN25BE,False,10.9,0.619,0.429,0.619,0.429,...,0.619,0.429,0.619,0.429,0.0,0.0,0.0,0.0,0.0,False
ADUR001B,Adur 001B,1.86762,BN25BE,BN25BE,False,11.9,0.619,0.429,0.619,0.429,...,0.619,0.429,0.619,0.429,0.0,0.0,0.0,0.0,0.0,False
ADUR001C,Adur 001C,2.24110,BN112DH,BN25BE,True,95.0,0.603,0.376,0.597,0.356,...,0.603,0.376,0.597,0.356,0.0,0.0,0.0,0.0,0.0,False


In [5]:
print(df.columns)

Index(['area_lower', 'Admissions_2025_Est', 'nearest_ivt_unit_base',
       'transfer_unit_base', 'transfer_required_base',
       'drip_ship_mt_travel_time_base', 'drip_ship_mrs_0-2_base',
       'lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'drip_ship_mrs_0-2_transferloss0!6_base',
       'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base',
       'nearest_ivt_unit_taunworc', 'transfer_unit_taunworc',
       'transfer_required_taunworc', 'drip_ship_mt_travel_time_taunworc',
       'drip_ship_mrs_0-2_taunworc', 'lvo_drip_ship_ivt_mt_mrs_0-2_taunworc',
       'drip_ship_mrs_0-2_transferloss0!6_taunworc',
       'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_taunworc',
       'change_in_drip_ship_mt_travel_time', 'change_in_drip_ship_mrs_0-2',
       'change_in_lvo_drip_ship_ivt_mt_mrs_0-2',
       'change_in_drip_ship_mrs_0-2_transferloss0!6',
       'change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6', 'changed'],
      dtype='str')


Gather columns by type and without tailing scenario names (`_base`, `_taunworc`):

In [6]:
col_lsoa11nm = 'area_lower'
col_admissions = 'Admissions_2025_Est'
cols_unit_postcodes = ['nearest_ivt_unit', 'transfer_unit']
cols_transfer = ['transfer_required']
cols_travel_time = ['drip_ship_mt_travel_time']
cols_outcome = [
    'drip_ship_mrs_0-2',
    'lvo_drip_ship_ivt_mt_mrs_0-2',
    'drip_ship_mrs_0-2_transferloss0!6',
    'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6',
]
cols_change = [
    'change_in_drip_ship_mt_travel_time',
    'change_in_drip_ship_mrs_0-2',
    'change_in_lvo_drip_ship_ivt_mt_mrs_0-2',
    'change_in_drip_ship_mrs_0-2_transferloss0!6',
    'change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6',
    'changed'
]

cols_shared = [col_lsoa11nm, col_admissions] + cols_change

### Hospital reference

In [7]:
# Load hospital coordinates from CSV and create a GeoDataFrame
p = os.path.join(dir_data, file_hospitals)
df_hospitals = pd.read_csv(p, index_col='Postcode')

# Only keep hospitals in England:
df_hospitals = df_hospitals.loc[df_hospitals["Country"] == "England"]

In [8]:
display(df_hospitals.head(3))

,SSNAP name,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT
Postcode,,,,,,,,,,,
OX39DU,John Radcliffe Hospital,England,"Buckinghamshire, Oxford and Berkshire",453940,207531,-1.219806,51.763871,Oxford,E01028534,1,1
RG15AN,Royal Berkshire Hospital,England,"Buckinghamshire, Oxford and Berkshire",472411,172962,-0.959330,51.451022,Reading,E01016414,1,0
HP112TT,Wycombe General Hospital,England,"Buckinghamshire, Oxford and Berkshire",486386,192694,-0.753428,51.626442,High Wycombe,E01017839,1,0


In [9]:
print(len(df_hospitals))

131


## Define functions

Create some functions for the calculations.

In [10]:
def append_scenario_to_column_names(cols, scens):
    new_cols = [f'{c}_{s}' for c in cols for s in scens]
    return new_cols

def remove_scenario_from_column_names(cols, scens):
    new_cols = [c.replace(f'_{s}', '').replace(s, '')
                for c in cols for s in scens]
    return new_cols

In [11]:
def calculate_admissions_by_unit(
        df,
        col_admissions,
        cols_unit_postcodes,
    ):
    df_admissions = pd.concat(
        [df.groupby(col)[col_admissions].sum().rename(f'admissions_{col}')
         for col in cols_unit_postcodes],
        axis='columns'
    )
    df_admissions = df_admissions.fillna(0.0)
    df_admissions = df_admissions.round(1)
    return df_admissions

In [12]:
def calculate_numbers_of_thrombectomy(
        df_admissions,
        mt_rate,
        transfer_rates,
        df_unit_links
    ):
    """
    Calculate:
    + total number of people receiving MT at each MT unit.
    + total number of people at each IVT unit who receive a transfer
      for MT, dependent on transfer rate.

    For IVT units, calculate admissions x MT rate. For non-MT units,
    also multiply by transfer rate.
    """
    # Pick out MT-providing units:
    m = df_admissions['admissions_transfer_unit'] > 0.0  #.notna()
    # Calculate MT for each IVT unit:
    cols_nearest = [c for c in df_admissions.columns
                    if 'nearest_ivt_unit' in c]
    df_mt = df_admissions[cols_nearest].copy() * mt_rate
    for t in transfer_rates:
        if t != 1.0:
            s = '' if t == 1.0 else f'_transferloss{t}'.replace('.', '!')
            cols_new = [f'{c}{s}' for c in cols_nearest]
            df_mt[cols_new] = df_mt[cols_nearest].copy()
            # Only scale for non-MT units:
            df_mt.loc[~m, cols_new] *= t

    # Merge in transfer unit postcodes:
    df_mt = pd.concat((df_mt, df_unit_links), axis='columns')
    col_transfer_postcodes = 'transfer_unit'
    # Gather MT admissions by MT unit:
    df_mt_mtunit = df_mt.groupby(col_transfer_postcodes).sum()
    rename_dict = dict([(c, c.replace('nearest_ivt_unit', 'transfer_unit'))
                         for c in df_mt_mtunit.columns])
    df_mt_mtunit = df_mt_mtunit.rename(columns=rename_dict)

    # Gather all results:
    df_mt = pd.concat((df_mt, df_mt_mtunit), axis='columns')
    df_mt = df_mt.drop(df_unit_links.columns, axis='columns')
    df_mt = df_mt.rename(
        columns=dict([(c, f'mt_{c}') for c in df_mt.columns]))

    df_mt = df_mt.fillna(0.0)
    df_mt = df_mt.round(1)
    return df_mt

In [13]:
def calculate_weighted_outcomes_by_lsoa(df, col_admissions, cols_outcome):
    """LSOA-level"""
    list_weighted_outcomes = []
    for col in cols_outcome:
        s = df[col] * df[col_admissions]
        s = s.rename(f'admissions_weighted_{col}')
        list_weighted_outcomes.append(s)

    df_outcomes = pd.concat(list_weighted_outcomes, axis='columns')
    df_outcomes = df_outcomes.round(5)
    return df_outcomes

In [14]:
def calculate_outcomes_by_unit(
        df_outcomes,  # by LSOA
        df_admissions,  # by unit
        cols_unit_postcodes,
        cols_admissions_weighted_outcomes
    ):
    list_dfs_unit_outcomes = []
    for col in cols_unit_postcodes:  # only do nearest stroke unit
        df_unit_outcomes = df_outcomes[
            [col] + cols_admissions_weighted_outcomes].groupby(col).sum()
        rename_dict = dict([(c, f'{col}_{c}')
                            for c in df_unit_outcomes.columns])
        df_unit_outcomes = df_unit_outcomes.rename(columns=rename_dict)
        list_dfs_unit_outcomes.append(df_unit_outcomes.copy())
    df_unit_outcomes = pd.concat(list_dfs_unit_outcomes, axis='columns')
    
    # Divide outcomes by the direct admissions to each unit:
    def scale_outcomes_by_admissions(df_unit_outcomes, df_admissions, col):
        for unit in df_admissions.index:
            df_unit_outcomes.loc[unit] = (
                df_unit_outcomes.loc[unit] /
                df_admissions.loc[unit, col]
            )
        return df_unit_outcomes

    df_unit_outcomes = scale_outcomes_by_admissions(
        df_unit_outcomes.copy(),
        df_admissions,
        'admissions_nearest_ivt_unit'
    )
    df_unit_outcomes = df_unit_outcomes.round(5)
    return df_unit_outcomes

Gather these separate functions into a main one:

In [15]:
def main(
        df,
        scenario,
        mt_rate,
        transfer_rates,
        cols_shared,
        col_admissions,
        cols_unit_postcodes,
        cols_outcome,
    ):
    """
    """
    print(scenario)
    df_here = df[cols_shared + [c for c in df.columns if c.endswith(scenario)]]
    rename_dict = dict([(c, c.replace(f'_{scenario}', ''))
                        for c in df_here.columns])
    df_here = df_here.rename(columns=rename_dict)

    # Link each first unit to its IVT unit:
    df_unit_links = (
        df_here[['nearest_ivt_unit', 'transfer_unit']]
        .drop_duplicates(ignore_index=True)
        .set_index('nearest_ivt_unit')
        )
    # Get unit names and cities:
    df_unit_info = df_unit_links.copy()
    cols_unit_info = ['SSNAP name', 'Hospital_City']
    df_unit_info = pd.merge(
        df_unit_info, df_hospitals[cols_unit_info],
        left_index=True, right_index=True, how='left')
    df_unit_info = pd.merge(
        df_unit_info.reset_index(),
        df_hospitals[cols_unit_info].reset_index(),
        left_on='transfer_unit', right_on='Postcode', how='left',
        suffixes=(None, '_transfer_unit')
    ).drop('Postcode', axis='columns').set_index('nearest_ivt_unit')
    print('df_unit_info:', df_unit_info.columns)
    
    # Sum admissions to each unit by scenario
    df_admissions = calculate_admissions_by_unit(
        df_here, col_admissions, cols_unit_postcodes)
    print('df_admissions:', df_admissions.columns)
    
    # Calculate numbers of thrombectomy.
    # Scale total admissions by the MT rate and by the transfer rate
    # to reflect that not everybody receives MT.
    # Record one column for the total numbers of thrombectomies,
    # and another for the thrombectomies that were transferred in.
    df_mt = calculate_numbers_of_thrombectomy(
        df_admissions, mt_rate, transfer_rates, df_unit_links)
    print('df_mt:', df_mt.columns)
    
    # Admissions-weighted outcomes
    # First calculate admissions-weighted outcomes by LSOA:
    df_outcomes = calculate_weighted_outcomes_by_lsoa(
        df_here, col_admissions, cols_outcome)
    cols_admissions_weighted_outcomes = list(df_outcomes.columns)
    # Merge in unit info and admissions:
    df_outcomes = pd.concat(
        (df_here[cols_unit_postcodes], df_outcomes),
        axis='columns'
    )

    # Group by unit:
    df_unit_outcomes = calculate_outcomes_by_unit(
        df_outcomes,  # by LSOA
        df_admissions,  # by unit
        [cols_unit_postcodes[0]], # only do nearest stroke unit
        cols_admissions_weighted_outcomes
    )
    print('df_unit_outcomes:', df_unit_outcomes.columns)
    # Only keep LVO mRS 0-2 outcomes:

    # Gather all results:
    # Make sure all index names match:
    comparison_by_ivt_unit = pd.concat(
        (df_unit_info, df_admissions, df_mt, df_unit_outcomes),
        axis='columns'
    )
    return comparison_by_ivt_unit

## Set up scenarios

Run calculations separately for each scenario, otherwise it's difficult to see what's happening.

In [16]:
mt_rate = 0.1
transfer_rates = [1.0, 0.6]  # should match existing results files.

comparison_by_ivt_unit_base = main(
    df,
    'base',
    mt_rate,
    transfer_rates,
    cols_shared,
    col_admissions,
    cols_unit_postcodes,
    cols_outcome,
)

comparison_by_ivt_unit_taunworc = main(
    df,
    'taunworc',
    mt_rate,
    transfer_rates,
    cols_shared,
    col_admissions,
    cols_unit_postcodes,
    cols_outcome,
)

base
df_unit_info: Index(['transfer_unit', 'SSNAP name', 'Hospital_City',
       'SSNAP name_transfer_unit', 'Hospital_City_transfer_unit'],
      dtype='str')
df_admissions: Index(['admissions_nearest_ivt_unit', 'admissions_transfer_unit'], dtype='str')
df_mt: Index(['mt_admissions_nearest_ivt_unit',
       'mt_admissions_nearest_ivt_unit_transferloss0!6',
       'mt_admissions_transfer_unit',
       'mt_admissions_transfer_unit_transferloss0!6'],
      dtype='str')
df_unit_outcomes: Index(['nearest_ivt_unit_admissions_weighted_drip_ship_mrs_0-2',
       'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
       'nearest_ivt_unit_admissions_weighted_drip_ship_mrs_0-2_transferloss0!6',
       'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6'],
      dtype='str')
taunworc
df_unit_info: Index(['transfer_unit', 'SSNAP name', 'Hospital_City',
       'SSNAP name_transfer_unit', 'Hospital_City_transfer_unit'],
      dtype='str')
df_admission

## Check for changes between scenarios

In [17]:
m = (comparison_by_ivt_unit_base.fillna(0.0) ==
     comparison_by_ivt_unit_taunworc.fillna(0.0)).all(axis='columns')

units_with_change = comparison_by_ivt_unit_base.loc[~m].index

In [18]:
units_with_change

Index(['BS105NB', 'B152TH', 'DY12HQ', 'WR51DD', 'GL537AN', 'EX314JB', 'PL68DH',
       'TA15DA', 'EX25DW', 'DT12JY', 'SO166YD', 'HR12ER'],
      dtype='str')

In [19]:
display(df_hospitals.loc[
    units_with_change, ['SSNAP name', 'Hospital_City']])

,SSNAP name,Hospital_City
BS105NB,Bristol and Weston SU,North Bristol
B152TH,Queen Elizabeth Hospital Edgbaston,Birmingham
DY12HQ,Russells Hall Hospital,Dudley
WR51DD,Worcestershire Royal Hospital,Worcester
GL537AN,Cheltenham & Gloucester Hospitals,Cheltenham
EX314JB,North Devon District Hospital,Barnstaple
PL68DH,Derriford Hospital,Plymouth
TA15DA,Musgrove Park Hospital,Taunton
EX25DW,Royal Devon and Exeter Hospital,Exeter
DT12JY,Dorset County Hospital,Dorchester


See that the expected units have changed - ones near Taunton and Worcester.

## Gather and simplify info

Place all results in same dataframe:

In [20]:
rename_base = dict([(c, f'{c}_base')
                    for c in comparison_by_ivt_unit_base.columns])
rename_taunworc = dict([(c, f'{c}_taunworc')
                       for c in comparison_by_ivt_unit_taunworc.columns])

comparison_by_ivt_unit = pd.concat(
    (#df_ivt_units,
     (comparison_by_ivt_unit_base
         .reset_index()
         .rename(columns={'index': 'nearest_ivt_unit'})
         .set_index(['nearest_ivt_unit', 'SSNAP name', 'Hospital_City'])
         .rename(columns=rename_base)
     ),
    (comparison_by_ivt_unit_taunworc
         .reset_index()
         .rename(columns={'index': 'nearest_ivt_unit'})
         .set_index(['nearest_ivt_unit', 'SSNAP name', 'Hospital_City'])
         .rename(columns=rename_taunworc)
     )
    ),
    axis='columns'
).sort_index().reset_index().set_index('nearest_ivt_unit')

In [21]:
comparison_by_ivt_unit.columns

Index(['SSNAP name', 'Hospital_City', 'transfer_unit_base',
       'SSNAP name_transfer_unit_base', 'Hospital_City_transfer_unit_base',
       'admissions_nearest_ivt_unit_base', 'admissions_transfer_unit_base',
       'mt_admissions_nearest_ivt_unit_base',
       'mt_admissions_nearest_ivt_unit_transferloss0!6_base',
       'mt_admissions_transfer_unit_base',
       'mt_admissions_transfer_unit_transferloss0!6_base',
       'nearest_ivt_unit_admissions_weighted_drip_ship_mrs_0-2_base',
       'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'nearest_ivt_unit_admissions_weighted_drip_ship_mrs_0-2_transferloss0!6_base',
       'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base',
       'transfer_unit_taunworc', 'SSNAP name_transfer_unit_taunworc',
       'Hospital_City_transfer_unit_taunworc',
       'admissions_nearest_ivt_unit_taunworc',
       'admissions_transfer_unit_taunworc',
       'mt_admissions_nearest_ivt_un

Some extra calculations:

In [22]:
# Changes between scenarios
comparison_by_ivt_unit['change_in_first_admissions'] = (
    comparison_by_ivt_unit['admissions_nearest_ivt_unit_taunworc']
    - comparison_by_ivt_unit['admissions_nearest_ivt_unit_base']
).round(1)
comparison_by_ivt_unit['change_in_all_MT_admissions'] = (
    comparison_by_ivt_unit['mt_admissions_transfer_unit_taunworc']
    - comparison_by_ivt_unit['mt_admissions_transfer_unit_base']
).round(1)
comparison_by_ivt_unit['change_in_outcome_lvo_mrs_0-2'] = (
    comparison_by_ivt_unit['nearest_ivt_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_taunworc']
    - comparison_by_ivt_unit['nearest_ivt_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base']
).round(5)

In [23]:
comparison_by_ivt_unit[['admissions_nearest_ivt_unit_taunworc',
    'admissions_nearest_ivt_unit_base', 'change_in_first_admissions']]

,admissions_nearest_ivt_unit_taunworc,admissions_nearest_ivt_unit_base,change_in_first_admissions
nearest_ivt_unit,,,
B152TH,861.1,861.1,0.0
B662QT,1799.4,1799.4,0.0
BA13NG,853.3,853.3,0.0
BB23HH,810.1,810.1,0.0
BD96RJ,1024.9,1024.9,0.0
...,...,...,...
WD180HB,1208.1,1208.1,0.0
WF14DG,1065.7,1065.7,0.0
WR51DD,705.4,700.6,4.8


In [24]:
# Find MT units with a change in total admissions
mt_units_with_change = comparison_by_ivt_unit.loc[
    comparison_by_ivt_unit['change_in_all_MT_admissions'] != 0
].index.tolist()

# Indicate whether each MT unit's admissions changed from any IVT unit
comparison_by_ivt_unit['mt_unit_admissions_changed_by_any_ivt_unit'] = (
    comparison_by_ivt_unit['transfer_unit_base'].isin(mt_units_with_change)
)

In [25]:
mt_units_with_change

['B152TH', 'BS105NB', 'PL68DH', 'SO166YD', 'TA15DA', 'WR51DD']

In [26]:
p = os.path.join(dir_output, 'comparison_by_ivt_unit_base_taunworc.csv')
comparison_by_ivt_unit.to_csv(p, index=True)#, index_label='IVT_Unit_Postcode', index=True)